# Base de análise estatística (Python)

Este notebook é um **ponto de partida**: mostra, passo a passo, como aplicar
**Shapiro-Wilk → Levene → ANOVA/Kruskal → Tukey/Dunn → FDR-BH → Vargha-Delaney A12**
e imprime os resultados. Ele **não** gera um relatório pronto.

**Premissa:** o CSV tem **uma persona** e **um sistema** (o pipeline gera um CSV por
sistema), com **8 configurações** e **n = 5 execuções** por configuração. Cada linha
é uma execução; o par `(config, repeticao)` deve ser único e a geração deve ser a final.

**Como usar:** edite `CAMINHO_CSV` e `INDICADOR` na seção 0 e rode as células em ordem.

**Requisitos:** `numpy`, `scipy`, `pandas`, `matplotlib`.


## 0. Configuração — edite aqui

In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Caminho do CSV de UM sistema (ex.: por_sistema/spring-petclinic.csv).
CAMINHO_CSV = "analise_estudo/resultados/por_sistema/spring-petclinic.csv"

# Indicador: "hv", "igd_plus", "ed_media", "ed_min" (e opcionalmente "spacing").
INDICADOR = "hv"

ALFA = 0.05
# Use True para adotar Kruskal-Wallis como escolha conservadora (n = 5).
FORCAR_KRUSKAL = False
EXIGIR_N = 5


## 1. Leitura e inspeção dos dados

O CSV tem **uma linha por execução** (8 configurações × 5 repetições = 40 linhas).
Conferimos as colunas, a persona, o sistema, a unicidade de `(config, repeticao)` e a
geração. Se houver mais de uma geração, mantemos a final (maior).


In [ ]:
caminho = Path(CAMINHO_CSV).expanduser()
if not caminho.exists():
    raise FileNotFoundError(f"CSV não encontrado: {caminho}")

df = pd.read_csv(caminho)

personas = df["persona"].dropna().unique()
sistemas = df["sistema"].dropna().unique()
if len(personas) != 1:
    raise ValueError(f"o CSV deve conter 1 persona; encontradas: {list(personas)}")
if len(sistemas) != 1:
    raise ValueError(f"o CSV deve conter 1 sistema; encontrados: {list(sistemas)}")

persona, sistema = personas[0], sistemas[0]

# Uma linha por execução: (config, repeticao) não pode repetir.
if df.duplicated(subset=["config", "repeticao"]).any():
    raise ValueError("há (config, repeticao) duplicado; o CSV deve ter 1 linha por execução.")

# Se houver mais de uma geração, manter a final (maior) e avisar.
if "geracao" in df.columns and df["geracao"].notna().any():
    geracoes = sorted(df["geracao"].dropna().unique())
    if len(geracoes) > 1:
        print(f"[aviso] múltiplas gerações {geracoes}; mantendo a maior (final).")
        df = df[df["geracao"] == max(geracoes)].copy()

print(f"Persona: {persona}")
print(f"Sistema: {sistema}")
print(f"Indicador: {INDICADOR}\n")

print("Primeiras linhas (colunas de interesse):")
print(df[["config", "repeticao", "geracao", "completo", INDICADOR]].head(10).to_string(index=False))


## 2. Preparação dos dados

Filtramos execuções completas, com fronteira não vazia (`n_pontos > 0`) e com valor
numérico. Mostramos quantas linhas cada configuração perdeu e conferimos a premissa
de **n = 5** por configuração.


In [ ]:
d = df.copy()
d["completo"] = d["completo"].astype(str).str.upper().isin(["TRUE", "1", "SIM"])
d = d[d["completo"] & (d["n_pontos"] > 0)].copy()

d[INDICADOR] = pd.to_numeric(d[INDICADOR], errors="coerce")
d = d.dropna(subset=[INDICADOR]).copy()

# Quadro de linhas brutas x válidas por configuração (evita exclusão silenciosa).
quadro = df.groupby("config").size().rename("linhas_brutas").to_frame()
quadro = quadro.join(d.groupby("config").size().rename("linhas_validas"), how="left")
quadro["linhas_validas"] = quadro["linhas_validas"].fillna(0).astype(int)
quadro["excluidas"] = quadro["linhas_brutas"] - quadro["linhas_validas"]
print(quadro.to_string())

sem_linhas = quadro.index[quadro["linhas_validas"] == 0].tolist()
if sem_linhas:
    print(f"\n[ATENÇÃO] configurações sem linhas válidas: {sem_linhas}")
if len(d) == 0:
    raise ValueError("nenhuma linha válida após os filtros")

niveis = sorted(d["config"].unique())
if len(niveis) < 2:
    raise ValueError("menos de 2 configurações com dados; não há comparação possível")

n_por_config = d.groupby("config").size().sort_index()
print("\nValores válidos por configuração (esperado: 5):")
print(n_por_config.to_string())

if (n_por_config < 3).any():
    print("\n[ATENÇÃO] alguma configuração tem n < 3: os testes e o A12 são pouco confiáveis.")
if (n_por_config != EXIGIR_N).any():
    print(f"\n[aviso] alguma configuração não tem n = {EXIGIR_N}.")

grupos = [d.loc[d["config"] == k, INDICADOR].to_numpy() for k in niveis]
print(f"\nConfigurações: {len(niveis)}")


## 3. Análise descritiva

In [ ]:
resumo = (
    d.groupby("config")[INDICADOR]
     .agg(["count", "mean", "median", "std", "min", "max"])
     .round(4)
)
print(resumo.to_string())

fig, ax = plt.subplots(figsize=(11, 4))
ax.boxplot(grupos)
ax.set_xticklabels(niveis, rotation=90)
ax.set_ylabel(INDICADOR)
ax.set_title(f"{persona} | {sistema} | {INDICADOR}")
plt.tight_layout()
plt.show()

# Q-Q plots por configuração (com n = 5 são apenas indicativos).
ncols = min(4, len(niveis))
nrows = int(np.ceil(len(niveis) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
for ax, k in zip(np.ravel(axes), niveis):
    stats.probplot(d.loc[d["config"] == k, INDICADOR], dist="norm", plot=ax)
    ax.set_title(k, fontsize=8)
for ax in np.ravel(axes)[len(niveis):]:
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.close("all")


## 4. Pressuposto de normalidade (Shapiro-Wilk)

`H0`: os dados são normais. `p > 0,05` → normalidade plausível.
Com n = 5, o teste tem baixo poder; interprete com cautela.


In [ ]:
linhas = []
for k in niveis:
    x = d.loc[d["config"] == k, INDICADOR]
    if len(x) < 3 or x.nunique() < 2:
        w, p = np.nan, np.nan
    else:
        w, p = stats.shapiro(x)
    linhas.append({"config": k, "n": len(x), "W": w, "p": p})

shapiro_df = pd.DataFrame(linhas)
print(shapiro_df.round(6).to_string(index=False))

normalidade_ok = bool((shapiro_df["p"] > ALFA).all())
print(f"\nNormalidade plausível em todas as configurações? {normalidade_ok}")


## 5. Pressuposto de homogeneidade das variâncias (Levene)

`H0`: variâncias homogêneas. `p > 0,05` → homogêneas.


In [ ]:
levene = stats.levene(*grupos, center="median")
print(f"Levene: W = {levene.statistic:.4f}, p = {levene.pvalue:.4g}")

levene_ok = bool(levene.pvalue > ALFA)
print(f"Variâncias homogêneas? {levene_ok}")


## 6. Teste global

Se **ambos** os pressupostos forem atendidos → **ANOVA de uma via**.
Caso contrário (ou `FORCAR_KRUSKAL = True`) → **Kruskal-Wallis**.
Se `p > 0,05`, não há diferença significativa e **o pós-hoc se encerra** (o A12 continua).


In [ ]:
if d[INDICADOR].nunique() < 2:
    print("[aviso] indicador constante: teste global indefinido (p = 1); A12 continua.")
    p_global = 1.0
    significativo = False
else:
    usar_anova = normalidade_ok and levene_ok and not FORCAR_KRUSKAL
    print("Teste global:", "ANOVA de uma via" if usar_anova else "Kruskal-Wallis")

    if usar_anova:
        global_teste = stats.f_oneway(*grupos)
        print(f"F = {global_teste.statistic:.4f}, p = {global_teste.pvalue:.4g}")
    else:
        global_teste = stats.kruskal(*grupos)
        print(f"H = {global_teste.statistic:.4f}, p = {global_teste.pvalue:.4g}")

    p_global = float(global_teste.pvalue)
    significativo = p_global <= ALFA
    print(f"\nDiferença significativa? {significativo} (p = {p_global:.4g})")


## 7. Pós-hoc + correção FDR-BH

Somente se o teste global for significativo:
- após ANOVA → **Tukey HSD** (o p do Tukey já é ajustado para as 28 comparações);
- após Kruskal → **Dunn** (p bruto, com correção de empates).

Os p-valores do pós-hoc são então corrigidos por **Benjamini-Hochberg (FDR)**. No caso
do Tukey, o BH é uma **camada extra conservadora** (dupla correção) — a coluna
`p_teste` já vem ajustada pelo Tukey.


In [ ]:
def bh_fdr(pvals):
    """Ajuste de Benjamini-Hochberg (FDR), preservando NaN."""
    p = np.asarray(pvals, dtype=float)
    ajustado = np.full(p.shape, np.nan, dtype=float)
    mask = np.isfinite(p)
    if mask.any():
        pf = p[mask]
        n = len(pf)
        ordem = np.argsort(pf)
        ranqueado = pf[ordem] * n / (np.arange(n) + 1)
        ranqueado = np.minimum.accumulate(ranqueado[::-1])[::-1]
        bloco = np.empty(n, dtype=float)
        bloco[ordem] = np.clip(ranqueado, 0.0, 1.0)
        ajustado[mask] = bloco
    return ajustado


def dunn_pairwise(valores, rotulos):
    """Dunn (1964) com correção de empates; níveis derivados dos rótulos."""
    valores = np.asarray(valores, dtype=float)
    rotulos = np.asarray(rotulos)
    N = len(valores)
    ranks = stats.rankdata(valores)
    niveis_local = sorted(set(rotulos))

    _, contagens = np.unique(valores, return_counts=True)
    T = np.sum(contagens**3 - contagens)
    sigma2 = N * (N + 1) / 12 - T / (12 * (N - 1))

    linhas = []
    for i in range(len(niveis_local) - 1):
        for j in range(i + 1, len(niveis_local)):
            a, b = niveis_local[i], niveis_local[j]
            ra = ranks[rotulos == a]
            rb = ranks[rotulos == b]
            denom = np.sqrt(sigma2 * (1 / len(ra) + 1 / len(rb)))
            z = (ra.mean() - rb.mean()) / denom if denom > 0 else np.nan
            p = 2 * stats.norm.sf(abs(z)) if np.isfinite(z) else np.nan
            linhas.append({"comparacao": f"{a} - {b}", "z": z, "p_teste": p})
    return pd.DataFrame(linhas)


post = None
if significativo:
    if usar_anova:
        tuk = stats.tukey_hsd(*grupos)
        linhas = []
        for i, j in combinations(range(len(niveis)), 2):
            # p do Tukey já ajustado; stat = média_i - média_j.
            linhas.append({
                "comparacao": f"{niveis[i]} - {niveis[j]}",
                "diferenca": tuk.statistic[i, j],
                "p_teste": tuk.pvalue[i, j],
            })
        post = pd.DataFrame(linhas)
    else:
        post = dunn_pairwise(d[INDICADOR].to_numpy(), d["config"].to_numpy())

    post["p_bh"] = bh_fdr(post["p_teste"])
    post["significativo"] = post["p_bh"] <= ALFA
    print(post.round({"diferenca": 4, "z": 3, "p_teste": 6, "p_bh": 6}).to_string(index=False))
    print("\nPares significativos (p ajustado por BH <= ALFA):")
    print(post.loc[post["significativo"], ["comparacao", "p_bh"]].to_string(index=False))
else:
    print("Global não significativo: o pós-hoc se encerra (o A12 continua).")


## 8. Tamanho de efeito — Vargha-Delaney A12 (todos os pares)

`A12 = P(A melhor que B) + 0,5 · P(A = B)`.
Direção: **HV** maior melhor; **IGD+ / ED** menor melhor (tratada no cálculo).
Magnitude (convenção adotada): Desprezível (|A12−0,5| ≤ 0,06), Pequena (≤ 0,14),
Média (≤ 0,21), Grande (> 0,21).


In [ ]:
def a12(a, b):
    a = np.asarray(a, dtype=float)[:, None]
    b = np.asarray(b, dtype=float)[None, :]
    cmp = a - b
    return (np.sum(cmp > 0) + 0.5 * np.sum(cmp == 0)) / cmp.size


def orientar(x, indicador):
    x = np.asarray(x, dtype=float)
    return x if indicador == "hv" else -x


def classificar_a12(valor):
    if not np.isfinite(valor):
        return None
    d = abs(valor - 0.5)
    if d <= 0.06:
        return "Desprezível"
    if d <= 0.14:
        return "Pequena"
    if d <= 0.21:
        return "Média"
    return "Grande"


linhas = []
for a, b in combinations(niveis, 2):
    xa = orientar(d.loc[d["config"] == a, INDICADOR], INDICADOR)
    xb = orientar(d.loc[d["config"] == b, INDICADOR], INDICADOR)
    valor = a12(xa, xb)
    if valor > 0.5:
        direcao = "A melhor que B"
    elif valor < 0.5:
        direcao = "B melhor que A"
    else:
        direcao = "Equivalentes"
    linhas.append({
        "config_A": a, "config_B": b, "A12": valor,
        "direcao": direcao, "magnitude": classificar_a12(valor),
    })

efeito = pd.DataFrame(linhas)
print(efeito.to_string(index=False))
print("\nEfeitos grandes:")
print(efeito.loc[efeito["magnitude"] == "Grande"].to_string(index=False))


## 9. Interpretação

- **Shapiro**: `p > 0,05` → normalidade plausível (com n = 5, baixo poder).
- **Levene**: `p > 0,05` → variâncias homogêneas.
- **Global**: `p > 0,05` → sem diferença; `p ≤ 0,05` → ao menos uma configuração difere.
- **Pós-hoc**: significância sobre o **p ajustado (BH)**. No Tukey, `p_teste` já é
  ajustado e o BH é uma camada extra conservadora.
- **A12**: `A12 > 0,5` → A tende a superar B; `A12 < 0,5` → B tende a superar A.
  - Com n = 5, o A12 é discreto (múltiplos de 0,04) e aproximado.
- **Limitação**: com 28 comparações e n = 5, espera-se que poucos pares sobrevivam ao FDR-BH.

**Atenção à independência:** se as 5 repetições forem do mesmo participante, são
medidas repetidas (não amostras independentes); ANOVA/Kruskal/A12 podem superestimar
a precisão. **Múltiplos indicadores:** analisar HV, IGD+, ED separadamente não corrige
a família entre indicadores; pré-especifique ou aplique correção adicional.


## 10. (Opcional) salvar resultados

O notebook **não** salva nada automaticamente. Se quiser guardar, descomente e rode.


In [ ]:
# from pathlib import Path
# out = Path("resultados_notebook")
# out.mkdir(exist_ok=True)
# shapiro_df.to_csv(out / "shapiro.csv", index=False)
# efeito.to_csv(out / "a12.csv", index=False)
# if post is not None:
#     post.to_csv(out / "posthoc.csv", index=False)
